In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

from utils.model import build_resnet50
from utils.checkpoint import load_checkpoint
from utils.dataloaders import create_dataloaders

from utils.metrics import evaluate_model

from utils.explainability import (
    generate_gradcam,
    generate_gradcam_plus_plus
)

from utils.integrated_gradients import (
    generate_integrated_gradients
)

from utils.evaluation import (
    normalize_map,
    threshold_map,
    resize_mask,
    compute_iou,
    pointing_game,
    heatmap_coverage,
    confidence_drop
)

from utils.evaluator import ExplainabilityEvaluator

In [3]:
if torch.backends.mps.is_available():
    device = torch.device("mps")

elif torch.cuda.is_available():
    device = torch.device("cuda")

else:
    device = torch.device("cpu")

print(device)

mps


In [4]:
model = build_resnet50()

model = load_checkpoint(
    model,
    "../checkpoints/best_iegef_resnet50.pth",
    device
)

model = model.to(device)

model.eval()

print("Best IEGEF model loaded successfully!")

Best IEGEF model loaded successfully!


In [5]:
dataset_path = "../datasets/COVID-19_Radiography_Dataset"

(
    train_loader,
    val_loader,
    test_loader,
    train_df,
    val_df,
    test_df
) = create_dataloaders(
    dataset_path=dataset_path,
    batch_size=1,
    load_masks=True
)

print(f"Test Images : {len(test_df)}")

Test Images : 3175


In [6]:
results, y_true, y_pred, y_prob = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device
)

In [7]:
print("=" * 60)

print(f"Accuracy  : {results['Accuracy']:.4f}")
print(f"Precision : {results['Precision']:.4f}")
print(f"Recall    : {results['Recall']:.4f}")
print(f"F1 Score  : {results['F1 Score']:.4f}")
print(f"AUC       : {results['AUC']:.4f}")

print("=" * 60)

Accuracy  : 0.9622
Precision : 0.9625
Recall    : 0.9622
F1 Score  : 0.9621
AUC       : 0.9935


In [8]:
print(results["Classification Report"])

              precision    recall  f1-score   support

           0     0.9853    0.9871    0.9862       542
           1     0.9491    0.9765    0.9626      1529
           2     0.9652    0.9224    0.9433       902
           3     0.9898    0.9653    0.9774       202

    accuracy                         0.9622      3175
   macro avg     0.9724    0.9628    0.9674      3175
weighted avg     0.9625    0.9622    0.9621      3175



In [9]:
print(results["Confusion Matrix"])

[[ 535    5    1    1]
 [   6 1493   29    1]
 [   0   70  832    0]
 [   2    5    0  195]]


In [10]:
evaluator = ExplainabilityEvaluator(
    model=model,
    test_loader=test_loader,
    device=device
)

In [11]:
gradcam_results = evaluator.evaluate_gradcam()

print(gradcam_results)

Evaluating Grad-CAM: 100%|██████████| 3175/3175 [03:22<00:00, 15.67it/s]


{'Method': 'Grad-CAM', 'Mean IoU': np.float64(0.234523967033939), 'Pointing Game': np.float64(0.7927559055118111), 'Heatmap Coverage': np.float64(0.6832298612034708)}


In [12]:
gradcampp_results = evaluator.evaluate_gradcampp()

print(gradcampp_results)

Evaluating Grad-CAM++: 100%|██████████| 3175/3175 [03:18<00:00, 16.03it/s]

{'Method': 'Grad-CAM++', 'Mean IoU': np.float64(0.2657771488950347), 'Pointing Game': np.float64(0.7804724409448819), 'Heatmap Coverage': np.float64(0.6711617416352305)}


In [13]:
ig_results = evaluator.evaluate_integrated_gradients()

print(ig_results)

Evaluating Integrated Gradients: 100%|██████████| 3175/3175 [37:14<00:00,  1.42it/s]

{'Method': 'Integrated Gradients', 'Mean IoU': np.float64(0.0006776081857317067), 'Pointing Game': np.float64(0.10960629921259843), 'Heatmap Coverage': np.float64(0.12572890939807818)}


In [14]:
evaluator.save_results(

    summary_file="../results/iegef_summary.csv",

    detailed_file="../results/iegef_image_results.csv"

)


Summary Results
                 Method  Mean IoU  Pointing Game  Heatmap Coverage
0              Grad-CAM  0.234524       0.792756          0.683230
1            Grad-CAM++  0.265777       0.780472          0.671162
2  Integrated Gradients  0.000678       0.109606          0.125729

Saved: ../results/iegef_summary.csv
Saved: ../results/iegef_image_results.csv


In [17]:
import pandas as pd

baseline = pd.read_csv("../results/baseline_summary.csv")
iegef = pd.read_csv("../results/iegef_summary.csv")

In [ ]:
comparison = pd.DataFrame({

    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "AUC",
        "Grad-CAM IoU",
        "Grad-CAM++ IoU",
        "Integrated Gradients IoU",
        "Grad-CAM Pointing Game",
        "Grad-CAM++ Pointing Game",
        "IG Pointing Game",
        "Grad-CAM Coverage",
        "Grad-CAM++ Coverage",
        "IG Coverage"
    ],

    "Baseline": [
        baseline_cls["Accuracy"],
        baseline_cls["Precision"],
        baseline_cls["Recall"],
        baseline_cls["F1 Score"],
        baseline_cls["AUC"],

        baseline.loc[baseline["Method"]=="Grad-CAM","Mean IoU"].iloc[0],
        baseline.loc[baseline["Method"]=="Grad-CAM++","Mean IoU"].iloc[0],
        baseline.loc[baseline["Method"]=="Integrated Gradients","Mean IoU"].iloc[0],

        baseline.loc[baseline["Method"]=="Grad-CAM","Pointing Game"].iloc[0],
        baseline.loc[baseline["Method"]=="Grad-CAM++","Pointing Game"].iloc[0],
        baseline.loc[baseline["Method"]=="Integrated Gradients","Pointing Game"].iloc[0],

        baseline.loc[baseline["Method"]=="Grad-CAM","Heatmap Coverage"].iloc[0],
        baseline.loc[baseline["Method"]=="Grad-CAM++","Heatmap Coverage"].iloc[0],
        baseline.loc[baseline["Method"]=="Integrated Gradients","Heatmap Coverage"].iloc[0],
    ],

    "IEGEF": [
        results["Accuracy"]*100,
        results["Precision"]*100,
        results["Recall"]*100,
        results["F1 Score"]*100,
        results["AUC"]*100,

        iegef.loc[iegef["Method"]=="Grad-CAM","Mean IoU"].iloc[0],
        iegef.loc[iegef["Method"]=="Grad-CAM++","Mean IoU"].iloc[0],
        iegef.loc[iegef["Method"]=="Integrated Gradients","Mean IoU"].iloc[0],

        iegef.loc[iegef["Method"]=="Grad-CAM","Pointing Game"].iloc[0],
        iegef.loc[iegef["Method"]=="Grad-CAM++","Pointing Game"].iloc[0],
        iegef.loc[iegef["Method"]=="Integrated Gradients","Pointing Game"].iloc[0],

        iegef.loc[iegef["Method"]=="Grad-CAM","Heatmap Coverage"].iloc[0],
        iegef.loc[iegef["Method"]=="Grad-CAM++","Heatmap Coverage"].iloc[0],
        iegef.loc[iegef["Method"]=="Integrated Gradients","Heatmap Coverage"].iloc[0],
    ]
})

comparison

,Metric,Baseline,IEGEF
0,Accuracy,96.2200,96.220472
1,Precision,96.2200,96.245972
2,Recall,96.2200,96.220472
3,F1,96.2200,96.209116
4,AUC,99.1900,99.354802
5,GradCAM IoU,0.1819,0.234524
6,GradCAM++ IoU,0.1917,0.265777
7,Integrated Gradients IoU,0.0012,0.000678
8,Pointing Game,0.7858,0.792756
9,Heatmap Coverage,0.6252,0.683230


In [16]:
comparison.to_csv(
    "../results/baseline_vs_iegef.csv",
    index=False
)